# Communicating with Plots: Themes and Layout

Labels, annotations, and scales polish a single figure.
This notebook covers the **frame** around the data: seaborn/matplotlib **themes** and **multi-panel layouts**.

**Learning objectives**

- Switch between common seaborn style contexts.
- Place titles, captions, and legends for publication.
- Combine several axes into one figure with `subplots` and `subplot_mosaic`.


In [ ]:
MPG_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/ggplot2/mpg.csv"
DIAMONDS_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/ggplot2/diamonds.csv"
PRESIDENTIAL_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/ggplot2/presidential.csv"

import textwrap

import numpy as np  # https://numpy.org/doc/
import pandas as pd  # https://pandas.pydata.org/docs/
import matplotlib.pyplot as plt  # https://matplotlib.org/stable/
import seaborn as sns  # https://seaborn.pydata.org/
from matplotlib import dates as mdates
from matplotlib import ticker

sns.set_theme()

mpg = pd.read_csv(MPG_URL, index_col=0)
diamonds = pd.read_csv(DIAMONDS_URL, index_col=0)
presidential = pd.read_csv(PRESIDENTIAL_URL, index_col=0, parse_dates=["start", "end"])


## Themes

Themes control non-data ink: background, grid lines, fonts, and legend boxes.

- In ggplot2 these are `theme_*()` functions; in Python, `sns.set_theme(style=...)`.
- Seaborn ships several styles: `darkgrid` (default), `whitegrid`, `white`, `ticks`, `dark`.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
with sns.axes_style("whitegrid"):
    sns.scatterplot(data=mpg, x="displ", y="hwy", hue="class", ax=ax)
    sns.regplot(data=mpg, x="displ", y="hwy", scatter=False, lowess=True, color="black", ax=ax)
ax.set_title("whitegrid style")
plt.show()


In [ ]:
styles = ["darkgrid", "whitegrid", "white", "ticks", "dark"]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flat

for ax, style in zip(axes, styles):
    with sns.axes_style(style):
        sns.barplot(data=mpg, x="class", y="hwy", ax=ax, errorbar=None)
    ax.set_title(style)
    ax.tick_params(axis="x", rotation=45)

axes[-1].axis("off")
plt.tight_layout()
plt.show()


You can nudge individual elements — bold title, caption, legend frame:


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=mpg, x="displ", y="hwy", hue="drv", ax=ax)
ax.set_title(
    "Larger engine sizes tend to have lower fuel economy",
    loc="left",
    fontweight="bold",
)
fig.text(0.0, 0.01, "Source: https://fueleconomy.gov", ha="left", fontsize=9)
legend = ax.legend(title="Drive train", loc="center right")
legend.get_frame().set_edgecolor("black")
plt.tight_layout()
plt.show()


## Layout

Combine related views in one figure so readers compare them side by side.

- Save each subplot on its own `Axes` object.
- Use `plt.subplots` for grids or `plt.subplot_mosaic` for irregular layouts.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.scatterplot(data=mpg, x="displ", y="hwy", ax=axes[0])
axes[0].set_title("Highway MPG vs engine size")

sns.boxplot(data=mpg, x="drv", y="hwy", ax=axes[1])
axes[1].set_title("Highway MPG by drive train")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

sns.scatterplot(data=mpg, x="displ", y="hwy", ax=axes[0, 0])
axes[0, 0].set_title("Engine size vs highway MPG")

sns.scatterplot(data=mpg, x="cty", y="hwy", ax=axes[0, 1])
axes[0, 1].set_title("City vs highway MPG")

sns.boxplot(data=mpg, x="drv", y="hwy", ax=axes[1, 0])
axes[1, 0].set_title("Highway MPG by drive train")

axes[1, 1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 2])

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])
ax5 = fig.add_subplot(gs[2:, :])

for ax, x, y, title in [
    (ax1, "drv", "cty", "City MPG"),
    (ax2, "drv", "hwy", "Highway MPG"),
]:
    sns.boxplot(data=mpg, x=x, y=y, hue="drv", dodge=False, legend=False, ax=ax)
    ax.set_title(title)

sns.kdeplot(data=mpg, x="cty", hue="drv", fill=True, alpha=0.4, ax=ax3)
ax3.set_title("City MPG density")

sns.kdeplot(data=mpg, x="hwy", hue="drv", fill=True, alpha=0.4, ax=ax4)
ax4.set_title("Highway MPG density")

sns.scatterplot(data=mpg, x="cty", y="hwy", hue="drv", ax=ax5)
ax5.set_title("City vs highway MPG")

handles, labels = ax5.get_legend_handles_labels()
fig.legend(handles, labels, title="Drive train", loc="upper center", ncol=3)
ax5.legend_.remove()

fig.suptitle(
    "City and highway mileage for cars with different drive trains",
    fontsize=14,
    y=0.98,
)
fig.text(0.99, 0.01, "Source: https://fueleconomy.gov", ha="right", fontsize=9)
plt.tight_layout()
plt.show()


## Summary

You can make graphics easier to read by layering:

- **Labels** — titles that state findings, axes with units, captions for sources.
- **Annotations** — text and arrows on the data layer.
- **Scales** — ticks, formatters, palettes, log axes, shared limits.
- **Themes** — background and typography.
- **Layout** — multiple panels in one figure.

Pair these mechanics with a book on visualization *thinking*, such as Albert Cairo's [*The Truthful Art*](https://www.amazon.com/gp/product/0321934075/).
